In [4]:
import os
import sys
import joblib
import pandas as pd
import numpy as np
sys.path.append('..')

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

os.makedirs('../models', exist_ok=True)
print('Imports OK')

Imports OK


Cargar dataset procesado y pipeline del Sprint 2

Dataset: se carga el raw y se hace el mismo split 80/20 con random_state=42 que usó el Rol 4, garantizando consistencia.

Pipeline: se carga preprocessor.pkl generado en 08_pipeline.ipynb. Solo se usará para transformar, no se re-fiteará

In [5]:
TARGET = 'default payment next month'

# Dataset raw
df = pd.read_csv('../data/raw/04-default_credit.csv')
X = df.drop(columns=[TARGET, 'ID'])
y = df[TARGET]

# Mismo split que Rol 4 (random_state=42, stratify=y)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f'Train: {X_train.shape[0]} filas | Test: {X_test.shape[0]} filas')
print(f'Distribución test: {y_test.value_counts().to_dict()}')

# Pipeline del Sprint 2 — congelado, solo transform()
preproc = joblib.load('../models/preprocessor.pkl')
print('\n✓ preprocessor.pkl cargado')

Train: 24000 filas | Test: 6000 filas
Distribución test: {0: 4673, 1: 1327}

✓ preprocessor.pkl cargado


Transformar datos con el preprocessor congelado

Se aplica transform() una sola vez. Todos los modelos usarán estos mismos datos transformados.

In [7]:
X_train_t = preproc.transform(X_train)
X_test_t  = preproc.transform(X_test)

print(f'X_train transformado: {X_train_t.shape}')
print(f'X_test  transformado: {X_test_t.shape}')
print('✓ Transformación aplicada sin re-fitear el preprocessor')

X_train transformado: (24000, 43)
X_test  transformado: (6000, 43)
✓ Transformación aplicada sin re-fitear el preprocessor


Entrenamiento de modelos baseline

Cada modelo se envuelve en un Pipeline([('clf', modelo)]) con parámetros por defecto, se entrena sobre X_train_t y se persiste con joblib.

Logistic Regression
¿Qué es? Modelo lineal de clasificación que estima la probabilidad de pertenencia a una clase mediante la función sigmoide.

¿Cómo funciona? Aprende un vector de pesos  w  minimizando la entropía cruzada. La predicción es  P(y=1|x)=σ(wTx+b) .

¿Por qué se incluye? Es el baseline interpretable por excelencia. Rápido, robusto y sirve como punto de referencia mínimo para comparar el resto de modelos.

In [4]:
pipe_lr = Pipeline([('clf', LogisticRegression(random_state=42, max_iter=1000))])
pipe_lr.fit(X_train_t, y_train)
joblib.dump(pipe_lr, '../models/baseline_lr.pkl')

y_pred = pipe_lr.predict(X_test_t)
y_prob = pipe_lr.predict_proba(X_test_t)[:, 1]
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print('✓ Guardado: models/baseline_lr.pkl')

Accuracy : 0.8003
F1 Score : 0.3793
ROC-AUC  : 0.7462
✓ Guardado: models/baseline_lr.pkl


Decision Tree
¿Qué es? Modelo jerárquico que particiona el espacio de features usando reglas del tipo feature <= umbral.

¿Cómo funciona? Construye el árbol de forma greedy eligiendo en cada nodo la división que maximiza la reducción de impureza Gini. Predice por la clase mayoritaria de la hoja donde cae cada ejemplo.

¿Por qué se incluye? Muy interpretable y visualizable. Captura relaciones no lineales y permite identificar qué features son más discriminativas

In [5]:
pipe_dt = Pipeline([('clf', DecisionTreeClassifier(random_state=42))])
pipe_dt.fit(X_train_t, y_train)
joblib.dump(pipe_dt, '../models/baseline_dt.pkl')

y_pred = pipe_dt.predict(X_test_t)
y_prob = pipe_dt.predict_proba(X_test_t)[:, 1]
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print('✓ Guardado: models/baseline_dt.pkl')

Accuracy : 0.7187
F1 Score : 0.3812
ROC-AUC  : 0.6025
✓ Guardado: models/baseline_dt.pkl


Random Forest

¿Qué es? Ensemble de múltiples árboles de decisión entrenados con Bagging y selección aleatoria de features en cada split.

¿Cómo funciona? Entrena N árboles sobre submuestras del dataset. La predicción final es la votación mayoritaria de todos los árboles. La aleatoriedad reduce la varianza respecto a un árbol individual.

¿Por qué se incluye? Uno de los modelos más robustos y versátiles. Generalmente supera al árbol individual, maneja bien muchas features y ofrece importancia de variables.

In [6]:
pipe_rf = Pipeline([('clf', RandomForestClassifier(random_state=42, n_jobs=-1))])
pipe_rf.fit(X_train_t, y_train)
joblib.dump(pipe_rf, '../models/baseline_rf.pkl')

y_pred = pipe_rf.predict(X_test_t)
y_prob = pipe_rf.predict_proba(X_test_t)[:, 1]
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print('✓ Guardado: models/baseline_rf.pkl')

Accuracy : 0.8093
F1 Score : 0.4521
ROC-AUC  : 0.7580
✓ Guardado: models/baseline_rf.pkl


 SVM (Support Vector Machine)
 
¿Qué es? Clasificador que busca el hiperplano de máximo margen entre clases. Con kernel RBF puede modelar fronteras de decisión no lineales.

¿Cómo funciona? Encuentra los vectores de soporte (puntos más cercanos al margen) que definen la frontera de decisión maximizando la distancia entre clases.

¿Por qué se incluye? Excelente en espacios de alta dimensionalidad. Requiere features escaladas, condición que ya garantiza el preprocessor del Sprint 2.

In [8]:
pipe_svm = Pipeline([('clf', SVC(probability=True, random_state=42))])
pipe_svm.fit(X_train_t, y_train)
joblib.dump(pipe_svm, '../models/baseline_svm.pkl')

y_pred = pipe_svm.predict(X_test_t)
y_prob = pipe_svm.predict_proba(X_test_t)[:, 1]
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print('✓ Guardado: models/baseline_svm.pkl')

Accuracy : 0.8133
F1 Score : 0.4595
ROC-AUC  : 0.7209
✓ Guardado: models/baseline_svm.pkl


KNN (K-Nearest Neighbors)

¿Qué es? Algoritmo basado en instancias que clasifica cada ejemplo según la clase de sus K vecinos más cercanos en el espacio de features.

¿Cómo funciona? Calcula la distancia euclidiana a todos los puntos de entrenamiento, selecciona los K más cercanos y predice por votación mayoritaria.

¿Por qué se incluye? Modelo no paramétrico que captura estructuras locales del dato. Muy sensible al escalado, por eso requiere que los datos vengan pre-escalados del preprocessor.

In [9]:
pipe_knn = Pipeline([('clf', KNeighborsClassifier(n_jobs=-1))])
pipe_knn.fit(X_train_t, y_train)
joblib.dump(pipe_knn, '../models/baseline_knn.pkl')

y_pred = pipe_knn.predict(X_test_t)
y_prob = pipe_knn.predict_proba(X_test_t)[:, 1]
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print('✓ Guardado: models/baseline_knn.pkl')

C:\Users\jesus\miniforge3\envs\proyecto_python\Lib\site-packages\threadpoolctl.py:1226: RuntimeWarning: 
Found Intel OpenMP ('libiomp') and LLVM OpenMP ('libomp') loaded at
the same time. Both libraries are known to be incompatible and this
can cause random crashes or deadlocks on Linux when loaded in the
same Python program.
Using threadpoolctl may cause crashes or deadlocks. For more
information and possible workarounds, please see
    https://github.com/joblib/threadpoolctl/blob/master/multiple_openmp.md

  warnings.warn(msg, RuntimeWarning)


Accuracy : 0.7900
F1 Score : 0.4167
ROC-AUC  : 0.7009
✓ Guardado: models/baseline_knn.pkl


Naive Bayes (Opcional)

¿Qué es? Clasificador probabilístico basado en el Teorema de Bayes con la suposición de independencia condicional entre features dado el target.

¿Cómo funciona? Estima  P(y)  y  P(xi|y)  con una distribución Gaussiana por cada feature. Predice la clase que maximiza  P(y)∏iP(xi|y) .

¿Por qué se incluye? Extremadamente rápido. Sirve como baseline probabilístico de referencia para verificar que modelos más complejos realmente aportan valor.

In [10]:
pipe_nb = Pipeline([('clf', GaussianNB())])
pipe_nb.fit(X_train_t, y_train)
joblib.dump(pipe_nb, '../models/baseline_nb.pkl')

y_pred = pipe_nb.predict(X_test_t)
y_prob = pipe_nb.predict_proba(X_test_t)[:, 1]
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print('✓ Guardado: models/baseline_nb.pkl')

Accuracy : 0.3720
F1 Score : 0.3962
ROC-AUC  : 0.7231
✓ Guardado: models/baseline_nb.pkl


Gradient Boosting (Opcional)

¿Qué es? Ensemble que construye árboles de forma secuencial, donde cada árbol corrige los errores del anterior.

¿Cómo funciona? Minimiza una función de pérdida de forma iterativa ajustando árboles pequeños sobre los residuos del modelo anterior. Combina muchos modelos débiles en uno fuerte.

¿Por qué se incluye? Generalmente uno de los mejores modelos en datos tabulares. Útil para establecer un techo de rendimiento antes de optimizar hiperparámetros.

In [11]:
pipe_gb = Pipeline([('clf', GradientBoostingClassifier(random_state=42))])
pipe_gb.fit(X_train_t, y_train)
joblib.dump(pipe_gb, '../models/baseline_gb.pkl')

y_pred = pipe_gb.predict(X_test_t)
y_prob = pipe_gb.predict_proba(X_test_t)[:, 1]
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, y_prob):.4f}')
print('✓ Guardado: models/baseline_gb.pkl')

Accuracy : 0.8183
F1 Score : 0.4719
ROC-AUC  : 0.7784
✓ Guardado: models/baseline_gb.pkl


Resumen comparativo

In [12]:
modelos = {
    'Logistic Regression':  joblib.load('../models/baseline_lr.pkl'),
    'Decision Tree':        joblib.load('../models/baseline_dt.pkl'),
    'Random Forest':        joblib.load('../models/baseline_rf.pkl'),
    'SVM':                  joblib.load('../models/baseline_svm.pkl'),
    'KNN':                  joblib.load('../models/baseline_knn.pkl'),
    'Naive Bayes':          joblib.load('../models/baseline_nb.pkl'),
    'Gradient Boosting':    joblib.load('../models/baseline_gb.pkl'),
}

rows = []
for nombre, pipeline in modelos.items():
    y_pred = pipeline.predict(X_test_t)
    y_prob = pipeline.predict_proba(X_test_t)[:, 1]
    rows.append({
        'Modelo':   nombre,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'F1 Score': round(f1_score(y_test, y_pred), 4),
        'ROC-AUC':  round(roc_auc_score(y_test, y_prob), 4),
    })

df_res = pd.DataFrame(rows).sort_values('ROC-AUC', ascending=False).reset_index(drop=True)
df_res

,Modelo,Accuracy,F1 Score,ROC-AUC
0,Gradient Boosting,0.8183,0.4719,0.7784
1,Random Forest,0.8093,0.4521,0.7580
2,Logistic Regression,0.8003,0.3793,0.7462
3,Naive Bayes,0.3720,0.3962,0.7231
4,SVM,0.8133,0.4595,0.7209
5,KNN,0.7900,0.4167,0.7009
6,Decision Tree,0.7187,0.3812,0.6025


Verificar modelos guardados

In [13]:
print('Modelos persistidos en models/:')
for f in sorted(os.listdir('../models')):
    if f.startswith('baseline'):
        size = os.path.getsize(f'../models/{f}') / 1024
        print(f'  {f:40s} {size:.1f} KB')

Modelos persistidos en models/:
  baseline_dt.pkl                          550.5 KB
  baseline_gb.pkl                          138.4 KB
  baseline_knn.pkl                         8250.9 KB
  baseline_lr.pkl                          1.3 KB
  baseline_nb.pkl                          2.2 KB
  baseline_rf.pkl                          52024.8 KB
  baseline_svm.pkl                         3841.5 KB
